# Feature Imputation with Gaussian and Gaussian Copula Imputers

This Notebook demonstrates how to use the Gaussian and Gaussian Copula imputers when computing Shapley values for model interpretability.

When explaining the prediction of a machine learning model $f: X \rightarrow Y$ with $M$ features, we need to define the utility $v(S)$ for a coalition of features $S \subseteq \{1, \ldots, M\}$.
Since most models cannot handle incomplete input vectors $x_S$, imputation is used to fill in the missing values $x_{\bar S}$, enabling model-agnostic explanations.

We will start by laying the theoretical basis for our imputers. For the actual usage in practice, jump to the section on going [from theory to code](#from-theory-to-code).

## Gaussian Imputation

$\def\bs{\boldsymbol}$
The approach for Gaussian imputation is to fill in the missing values $x_{\bar S}$ by using the statistical properties of the feature distribution. As described in [\[Aas21\]](../references.rst), we assume the feature vector $x$ follows a multivariate Gaussian distribution $\mathcal{N}_M(\bs{\mu}, \bs{\Sigma})$. The parameters $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ of the distribution are estimated using sample mean and covariance from training data.

### Conditional Gaussian Distributions

When the features are parted into observed set $S$ (i.e., a coalition) and unobserved set ${\bar S} = M \setminus S$, the conditional distribution of $x_{{\bar S}}$ given the observation $x_S= x_S ^*$ is again a multivariate Gaussian:

$$
    p(x_{{\bar S}}| x_S=x_s^*) = \mathcal{N}_{|{\bar S}|}(\bs{\mu}_{{\bar S}|S}, \bs{\Sigma}_{{\bar S}|S}),
$$

where the parameters $\bs{\mu_{\bar S | S}}$ and $\bs{\Sigma_{\bar S | S}}$ are calculated as follows:

$$
\boldsymbol{\mu}_{{\bar S}|S} = \boldsymbol{\mu}_{{\bar S}} + \bs{\Sigma}_{{\bar S}|S} \bs{\Sigma}_{S|S}^{-1} (x_S^* - \boldsymbol{\mu}_S) 
$$

$$
\bs{\Sigma}_{{\bar S}|S} = \bs{\Sigma}_{{\bar S}|{\bar S}} - \bs{\Sigma}_{{\bar S}|S} \bs{\Sigma}_{S|S}^{-1} \bs{\Sigma}_{S|{\bar S}}
$$

The vector $\boldsymbol{\mu}_{\bar{S}|S}$ is the _conditional mean_, representing the most likely value for $x_{\bar{S}}$ given $x_S = x^*_S$. It adjusts the average of the missing features based on their correlation with the observed features and how the observed values deviate from their mean. [\[Jul25\]](../references.rst)

### Monte Carlo Sampling 

Finally, we compute $v(S)$ by drawing $K$ samples $\{ x_{\bar S}^{(k)} \}_{k=1,\ldots,K}$ from the conditional distribution, evaluating the model on the completed feature vector $[x_{\bar S}^{(k)}, x_S]$ and averaging the resulting model predictions:

$$
v(S) = \frac{1}{K} \sum_{k=1}^{K} f(x_{\overline{S}}^{(k)}, x_S^*)
$$

Each sample $x_{\overline{S}}^{(k)}$ represents a statistically plausible completion of the missing features given the observed $x_S^*$. We call $x^{(k)}$ a completion because it _completes_ the partially observed instance $x_S^*$ into a full input for the model $f$.

### Gaussian Copula

The Gaussian copula method is used when the data does not follow a multivariate Gaussian distribution.
It allows us to model the dependencies among features even when they are not normally distributed.
This is achieved by transforming every feature column to the so-called _Gaussian space_, then imputing as usual, and finally transforming back to the original feature space.
The exact procedure is as follows:

1. Transform each feature $j$ to the Gaussian space by applying the empirical cumulative distribution function (eCDF) $\hat F_j$ followed by the standard normal quantile function:
    - $V_j = \Phi^{-1}(\hat{F}_j(x_j))$
2. Perform Gaussian imputation in this transformed space
3. Transform back to the original space using the empirical quantile function
    - $\hat{x}_j = \hat{F}_j^{-1}(\Phi(v_j))$

## From Theory to Code

Having established the theoretical foundation, we now turn to its implementation, showing how to compute the conditional mean and covariance in practice.
We are going to use the **Random Forest Regressor Model** and the **California Housing dataset** for this example.

_(This example is adapted from the Conditional Imputer notebook.)_

First, we need to import some modules, load the dataset and train the model.

In [ ]:
import shapiq
from shapiq.datasets import load_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X, y = load_california_housing()
X_train, X_test, y_train, y_test = train_test_split(
    X.values,
    y.values,
    test_size=0.25,
    random_state=42,
)
n_features = X_train.shape[1]

model = RandomForestRegressor(
    n_estimators=500,
    max_depth=n_features,
    max_features=2 / 3,
    max_samples=2 / 3,
    random_state=42,
)
# Train the model (this may take a while)
model.fit(X_train, y_train)

### Gaussian Imputation in Practice

The first imputer we are going to look at is the `GaussianImputer`.  It takes in the following key parameters:

- `model`: The model to explain as a callable function expecting data points as input and returning the model's predictions.
- `data`: The background data to use for the explainer as an `ndarray` of shape `(n_samples, n_features)`. Will be used to estimate parameters $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ of the feature distribution.
- `sample_size`: The number of Monte Carlo samples for imputation.

Let's create an imputer and use with the `TabularExplainer` provided by `shapiq`.

In [ ]:
from shapiq_student import GaussianImputer

# Create a GaussianImputer instance using the training set as background data
gaussian_imputer = GaussianImputer(
    # We need to pass the model's `predict` function here, since a Callable is expected
    model=model.predict,
    data=X_train,
    sample_size=100,
    random_state=42,
)

# Set up a `TabularExplainer` to use our imputer
explainer_gaussian = shapiq.TabularExplainer(
    model=model,
    data=X_train,
    index="SII",
    max_order=2,
    random_state=42,
    imputer=gaussian_imputer,
)

Now, we're ready to explain an instance from the test dataset.

In [ ]:
x_explain = X_test[100]
# Use the full budget `2**n_features` to get an exact explanation
gaussian_values = explainer_gaussian.explain(x_explain, budget=2**n_features, random_state=0)
print(gaussian_values)

We can visualize these results nicely in a Network Plot.

In [ ]:
fig, ax = shapiq.network_plot(
    interaction_values=gaussian_values,
    feature_names=X.columns,
)
ax.set_title("Shapley Interactions (Gaussian Imputation)", pad=20, fontsize="x-large");

### Gaussian Copula in Practice

Next up is the `GaussianCopulaImputer`, which handles non-normally distributed features.

The `GaussianCopulaImputer` is used pretty much the same as the `GaussianImputer`:

In [ ]:
from shapiq_student import GaussianCopulaImputer

# Create a GaussianCopulaImputer instance
copula_imputer = GaussianCopulaImputer(
    model=model.predict,
    data=X_train,
    sample_size=100,
    random_state=42,
)

# Set up a `TabularExplainer` to use our imputer
explainer_copula = shapiq.TabularExplainer(
    model=model,
    data=X_train,
    index="SII",
    max_order=2,
    random_state=42,
    imputer=copula_imputer,
)

Again, we can now explain an instance from the test dataset:

In [ ]:
iv_copula = explainer_copula.explain(x_explain, budget=2**n_features, random_state=0)
print(iv_copula)

Finally, we visualize our results.

In [ ]:
fig, ax = shapiq.network_plot(
    interaction_values=iv_copula,
    feature_names=X.columns,
)
ax.set_title("Shapley Interactions (Gaussian Copula Imputation)", pad=20, fontsize="x-large");

## Summary

Both imputation methods implemented here handle missing features via conditional imputation.
- Gaussian: Fast but assumes normality
- Gaussian copula: More general but computationally heavier
- The choice depends on data distribution!